# 03b - Training Model E* (Model E + fastsolv logS)

**Model E\*** is Model E with **one more input**: the logS of the molecule in the solvent, predicted by fastsolv from the SMILES. It is meant for molecules whose structure is known (case B of `src/workflow.py`).

- Same 16 features as Model E, plus `logS` (17 features).
- Same XGBoost settings, same train / test split (notebook 01), same cross-validation as notebook 04.
- The logS values come from `data/interim/fastsolv_predictions_all.xlsx`, built by `scripts/data_prep/fastsolv_all_molecules.py` (or the Colab version).

Run it after notebook 01. It:
1. adds the logS to the data of notebook 01;
2. compares Model E and Model E* by cross-validation, on exactly the same splits;
3. trains Model E* and saves it to `models/xgb_model_e_star_17features.joblib`, the model used by `src/workflow.py`.

**After adding a plate:** run `fastsolv_all_molecules.py` again first, so that the new molecules get a logS. This notebook stops if a molecule has none.

In [1]:
import sys
from pathlib import Path
# Find the project root (the folder that contains src/config.py) and make src/ importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import TRAIN, TEST, FASTSOLV_PREDICTIONS_ALL, MODEL_E_STAR, CV_PREDICTIONS_E_STAR, ensure_dir

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, balanced_accuracy_score

## Settings

- `TRAIN_ON_ALL = False` (default): the final model learns on the train set only, as in notebook 02.
- `TRAIN_ON_ALL = True`: the final model learns on **all** molecules (train + test). Use it for the model used by the workflow, once the evaluation is done.

The cross-validation below always uses all molecules, whatever this setting.

In [2]:
TRAIN_ON_ALL = False
N_FOLDS = 5        # same cross-validation as notebook 04
N_REPEATS = 3

In [3]:
# Model E: the 16 features in Arthur's order (cell 18). Model E*: the same + logS
cols_E = ['MolWt', 'Sol_Dielectric', 'Sol_Hansen_D', 'Sol_Hansen_P', 'Sol_Hansen_H']
vector_cols = [f'Matrix_Vector_{i}' for i in range(1, 11)]
xgb_cols = ['RT'] + cols_E + vector_cols
xgb_cols_star = xgb_cols + ['logS']

def make_model_E(scale_weight):
    """Arthur's Model E settings (rt_prediction_continued.ipynb, cell 18), used for both models."""
    return xgb.XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=4, subsample=0.8,
        colsample_bytree=0.5, colsample_bynode=0.9, gamma=.8, min_child_weight=10, reg_lambda=5, reg_alpha=1,
        scale_pos_weight=scale_weight, random_state=72, n_jobs=-1)

## Adding the logS to the data

`fastsolv_predictions_all.xlsx` has one row per molecule and one column `predicted_logS_<solvent>` per solvent. It is turned into one row per molecule x solvent, then joined to the rows of notebook 01 on `CAS` and `Solvent_Name`.

In [4]:
fastsolv = pd.read_excel(FASTSOLV_PREDICTIONS_ALL)
fastsolv['CAS'] = fastsolv['CAS'].astype(str).str.strip()
logs_cols = [c for c in fastsolv.columns if c.startswith('predicted_logS_') and 'stdev' not in c]
logS = fastsolv.melt(id_vars='CAS', value_vars=logs_cols, var_name='Solvent_Name', value_name='logS')
logS['Solvent_Name'] = logS['Solvent_Name'].str.replace('predicted_logS_', '', regex=False)

def add_logS(df):
    """Add the fastsolv logS of each molecule x solvent; stop if one is missing."""
    df = df.copy()
    df['CAS'] = df['CAS'].astype(str).str.strip()
    df = df.merge(logS, on=['CAS', 'Solvent_Name'], how='left')
    missing = df.loc[df['logS'].isna(), 'CAS'].unique()
    if len(missing):
        raise ValueError(f"{len(missing)} molecule(s) have no fastsolv logS, e.g. {list(missing[:5])}. "
                         "Run scripts/data_prep/fastsolv_all_molecules.py (or the Colab version) first.")
    return df

df_train = add_logS(pd.read_csv(TRAIN))
df_test = add_logS(pd.read_csv(TEST))
df_all = pd.concat([df_train, df_test], ignore_index=True)
print(f"{len(df_all)} rows, {df_all['CAS'].nunique()} molecules, all with a logS")

5560 rows, 194 molecules, all with a logS


## Cross-validation: Model E vs Model E*

Exactly the procedure of notebook 04 (molecules split into 5 groups, 3 different splits, same random seeds), run for both models on the same splits, so the comparison is fair.

- **Top-1:** the solvent ranked first (mean probability over the molecule's HPLC columns) dissolves the molecule.
- **Balanced accuracy:** soluble / insoluble answer per row, threshold 0.5.

In [5]:
def rank_per_molecule(df, proba):
    """One row per molecule x solvent: mean predicted probability over the molecule's HPLC columns."""
    g = df[['CAS', 'Solvent_Name', 'Soluble']].assign(p=proba)
    return (g.groupby(['CAS', 'Solvent_Name'], as_index=False).agg(p=('p', 'mean'), y=('Soluble', 'first'))
             .rename(columns={'Solvent_Name': 'Solvent'}))

def top1(ranking):
    """Share of molecules whose first-ranked solvent dissolves them."""
    best = ranking.loc[ranking.groupby('CAS')['p'].idxmax()]
    return best['y'].mean()

y = df_all['Soluble'].values
molecules = np.array(sorted(df_all['CAS'].unique()))
models = {'Model E': xgb_cols, 'Model E*': xgb_cols_star}

results, predictions_star = [], []
for rep in range(N_REPEATS):
    mol_order = np.random.RandomState(rep).permutation(len(molecules))
    fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[mol_order])
    for k in range(N_FOLDS):
        te = df_all['CAS'].isin(set(fold[fold == k].index)).values
        tr = ~te
        scale_weight = (tr.sum() - y[tr].sum()) / y[tr].sum()
        for name, cols in models.items():
            model = make_model_E(scale_weight).fit(df_all.loc[tr, cols], y[tr])
            proba = model.predict_proba(df_all.loc[te, cols])[:, 1]
            ranking = rank_per_molecule(df_all[te], proba)
            if name == 'Model E*':
                predictions_star.append(ranking.assign(rep=rep, fold=k))
            results.append({'model': name, 'split': rep + 1, 'fold': k + 1,
                            'Top1': top1(ranking),
                            'Balanced_accuracy': balanced_accuracy_score(y[te], (proba >= 0.5).astype(int)),
                            'Accuracy': accuracy_score(y[te], (proba >= 0.5).astype(int))})
        print(f"split {rep + 1} fold {k + 1} done")

results = pd.DataFrame(results)
pd.concat(predictions_star, ignore_index=True).to_csv(ensure_dir(CV_PREDICTIONS_E_STAR), index=False)
print(f"\nModel E* predictions saved: {CV_PREDICTIONS_E_STAR}")

split 1 fold 1 done


split 1 fold 2 done


split 1 fold 3 done


split 1 fold 4 done


split 1 fold 5 done


...

split 3 fold 1 done


split 3 fold 2 done


split 3 fold 3 done


split 3 fold 4 done


split 3 fold 5 done

Model E* predictions saved: /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/processed/cv_predictions_e_star.csv


In [6]:
summary = results.groupby('model')[['Top1', 'Balanced_accuracy', 'Accuracy']].agg(['mean', 'std'])
summary.columns = [f"{metric} {stat}" for metric, stat in summary.columns]
display((summary * 100).round(1))

,Top1 mean,Top1 std,Balanced_accuracy mean,Balanced_accuracy std,Accuracy mean,Accuracy std
model,,,,,,
Model E,78.2,5.2,62.1,4.3,64.9,3.6
Model E*,79.7,4.7,72.6,4.9,73.6,4.6


## Training the final Model E*

In [7]:
df_fit = df_all if TRAIN_ON_ALL else df_train
y_fit = df_fit['Soluble'].values
scale_weight = (len(y_fit) - y_fit.sum()) / y_fit.sum()
print(f"Training on {len(df_fit)} rows, {df_fit['CAS'].nunique()} molecules"
      f"{' (train + test)' if TRAIN_ON_ALL else ''} | scale_pos_weight: {scale_weight:.2f}")

model_E_star = make_model_E(scale_weight).fit(df_fit[xgb_cols_star], y_fit)

importance = pd.Series(model_E_star.feature_importances_, index=xgb_cols_star).sort_values(ascending=False)
display(importance.round(3).to_frame('importance').head(8))

Training on 3890 rows, 135 molecules | scale_pos_weight: 0.47


,importance
logS,0.262
Sol_Hansen_P,0.092
MolWt,0.087
Sol_Dielectric,0.081
Sol_Hansen_D,0.076
Sol_Hansen_H,0.074
RT,0.059
Matrix_Vector_9,0.035


## Saving

In [8]:
joblib.dump(model_E_star, ensure_dir(MODEL_E_STAR))
print(f"Model saved: {MODEL_E_STAR}")

Model saved: /home/coralie/workspace/ProjectII/solubility_prediction_nmr/models/xgb_model_e_star_17features.joblib
